In [1]:
import polars as pl
from datetime import datetime, timezone, timedelta

import openmeteo_requests
import requests_cache
from retry_requests import retry

In [2]:
# setup openmeteo api client
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    "latitude": 54.3435,
    "longitude": 10.115,
    "start_date": "2026-02-15",
    "end_date": "2026-03-01",
    "hourly": ["temperature_2m", "precipitation", "weather_code"],
}

In [ ]:
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]

# Process hourly data. The order of variables needs to be the same as requested.
# The following section is from the openmeteo api website. Originally the data was put into a pandas df. 
# We used chatgpt to change the code to create a polars df for the data.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(1).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(2).ValuesAsNumpy()

start = datetime.fromtimestamp(hourly.Time(), tz=timezone.utc)
end = datetime.fromtimestamp(hourly.TimeEnd(), tz=timezone.utc)
interval = timedelta(seconds=hourly.Interval())

date_range = pl.datetime_range(
    start,
    end,
    interval,
    closed="left",
    eager=True
)

In [4]:

# ---- Create Polars DataFrame ----
hourly_dataframe = pl.DataFrame({
    "date": date_range,
    "temperature_2m": hourly_temperature_2m,
    "precipitation": hourly_precipitation,
    "weather_code": hourly_weather_code,
})

print(hourly_dataframe)


shape: (360, 4)
┌─────────────────────────┬────────────────┬───────────────┬──────────────┐
│ date                    ┆ temperature_2m ┆ precipitation ┆ weather_code │
│ ---                     ┆ ---            ┆ ---           ┆ ---          │
│ datetime[μs, UTC]       ┆ f32            ┆ f32           ┆ f32          │
╞═════════════════════════╪════════════════╪═══════════════╪══════════════╡
│ 2026-02-15 00:00:00 UTC ┆ -12.95         ┆ 0.0           ┆ 3.0          │
│ 2026-02-15 01:00:00 UTC ┆ -12.8          ┆ 0.0           ┆ 3.0          │
│ 2026-02-15 02:00:00 UTC ┆ -12.25         ┆ 0.0           ┆ 3.0          │
│ 2026-02-15 03:00:00 UTC ┆ -12.4          ┆ 0.0           ┆ 3.0          │
│ 2026-02-15 04:00:00 UTC ┆ -12.5          ┆ 0.0           ┆ 3.0          │
│ …                       ┆ …              ┆ …             ┆ …            │
│ 2026-03-01 19:00:00 UTC ┆ 8.15           ┆ 0.0           ┆ 3.0          │
│ 2026-03-01 20:00:00 UTC ┆ 7.85           ┆ 0.0           ┆ 2.0        